In [1]:
import sys
from pathlib import Path

# Get parent directory (Thesis-Edvin)
sys.path.append(str(Path.cwd().parent))

In [2]:
from utils import *
from datasets import load_dataset
from rag import *

In [3]:
gpt_pred = load_dataset("Eathus/github-issues-vul-detection-gpt-few-results-final-best2", split="test")
gpt_pred_df = gpt_pred.to_pandas()
gpt_pred_df = gpt_pred_df[~gpt_pred_df.duplicated(subset="issue_github_id", keep=False)]

In [4]:
true_pos = gpt_pred_df[gpt_pred_df.gpt_is_relevant & ~gpt_pred_df.cve_id.isna()]
false_pos = gpt_pred_df[gpt_pred_df.gpt_is_relevant & gpt_pred_df.cve_id.isna()]
false_neg = gpt_pred_df[~gpt_pred_df.gpt_is_relevant & ~gpt_pred_df.cve_id.isna()]
all_true = gpt_pred_df[~gpt_pred_df.cve_id.isna()]

print("true pos:", len(true_pos))
print("false pos:", len(false_pos))
print("false neg:", len(false_neg))
print("all true:", len(all_true))
print("all", len(gpt_pred_df))

true pos: 296
false pos: 305
false neg: 10
all true: 306
all 1763


In [5]:
cwes = load_dataset("Eathus/cwe_view1000_list_gpt_few_cwe_desc_final_best", split="train")
cwes_df = cwes.to_pandas()

# View CWE-1000

In [6]:
cwes_dict = cwes_df.to_dict('records')
documents_light = [
    create_ordered_cwe_document(weakness, old=True) for weakness in cwes_dict
]
document_dict = {doc.metadata["CWE_ID"]: doc for doc in documents_light}
documents = [
    Document(
        page_content=cwe.gpt_cwe_description,
        metadata=document_dict[cwe.ID].metadata,  # Original metadata
    )
    for _, cwe in cwes_df.iterrows()
]

allowed_usage = ["Allowed", "Allowed-with-Review", "Discouraged"]
documents = filter_cwe_docs_usage(documents, allowed_usage)

documents = split_docs(documents, echo=True, max_tokens=800)

Original: 881 docs
After splitting: 901 docs


In [7]:
vectorstore = create_vectorstore(documents, "tmp/faiss_enhanced_kb_index")

Loading existing vector store...


In [8]:
ranges = [20]
for k in ranges :
    rag_labeled_df = bare_rag_label(
        true_pos, vectorstore, match_col="gpt_vulnerability", k=k
    )
    evaluate_rag(rag_labeled_df, col='rag_label', save=True, md_filename=f'tmp/enhanced_kb_results/classification_report_top-{k}.md')


100%|██████████| 296/296 [00:11<00:00, 25.65it/s]


✅ Markdown report saved to: tmp/enhanced_kb_results/classification_report_top-20.md


In [11]:
from datasets import Dataset

# test
rag_labeled_df = rag_labeled_df.drop('__index_level_0__', axis=1)
ds = Dataset.from_pandas(rag_labeled_df)
ds.push_to_hub("Eathus/github-issues-vul-label-rag-results2", split="test")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

CommitInfo(commit_url='https://huggingface.co/datasets/Eathus/github-issues-vul-label-rag-results2/commit/f8498f136e9d9a3c426ddcea26df52a1c3988129', commit_message='Upload dataset', commit_description='', oid='f8498f136e9d9a3c426ddcea26df52a1c3988129', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Eathus/github-issues-vul-label-rag-results2', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Eathus/github-issues-vul-label-rag-results2'), pr_revision=None, pr_num=None)

# View CWE-1003

In [9]:
view1003 = load_dataset("Eathus/cwe_view1003_raw_list", split="train")
view1003_df = view1003.to_pandas()

In [10]:
documents1003 = [doc for doc in documents if doc.metadata['CWE_ID'] in view1003_df['ID'].tolist()]

In [11]:
vectorstore1003 = create_vectorstore(documents1003, "tmp/faiss_enhanced_kb1003_index")

Loading existing vector store...


In [12]:
ranges = [1, 5, 10, 15, 20]
for k in ranges :
    rag_labeled_df = bare_rag_label(
        true_pos, vectorstore1003, match_col="gpt_vulnerability", k=k
    )
    evaluate_rag(rag_labeled_df, col='rag_label', save=True, md_filename=f'tmp/enhanced_kb_results/classification_report1003_top-{k}.md')

100%|██████████| 1264/1264 [01:03<00:00, 19.76it/s]



✅ Markdown report saved to: tmp/enhanced_kb_results/classification_report1003_top-1.md


100%|██████████| 1264/1264 [01:07<00:00, 18.65it/s]



✅ Markdown report saved to: tmp/enhanced_kb_results/classification_report1003_top-5.md


100%|██████████| 1264/1264 [01:09<00:00, 18.20it/s]



✅ Markdown report saved to: tmp/enhanced_kb_results/classification_report1003_top-10.md


100%|██████████| 1264/1264 [01:12<00:00, 17.53it/s]



✅ Markdown report saved to: tmp/enhanced_kb_results/classification_report1003_top-15.md


100%|██████████| 1264/1264 [01:07<00:00, 18.61it/s]



✅ Markdown report saved to: tmp/enhanced_kb_results/classification_report1003_top-20.md
